In [1]:
from typing import List
from pydantic import BaseModel,Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json

In [5]:
import requests

url = "http://100.108.110.17:8000/v1/models"

r = requests.get(url)

print(r.status_code)
print(r.text)

200
{"object":"list","data":[{"id":"Qwen/Qwen3.6-27B-FP8","object":"model","created":1779766147,"owned_by":"vllm","root":"/models/aeon-ultimate","parent":null,"max_model_len":200000,"permission":[{"id":"modelperm-b6dd846629cfa371","object":"model_permission","created":1779766147,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]},{"id":"qwen36-ultimate","object":"model","created":1779766147,"owned_by":"vllm","root":"/models/aeon-ultimate","parent":null,"max_model_len":200000,"permission":[{"id":"modelperm-88daa4ed5d6b16b9","object":"model_permission","created":1779766147,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]},{"id":"aeon-ultimate-xs","object":"model","created":1779766147,"owned_by":

In [2]:
llm = ChatOpenAI(model="Qwen/Qwen3.6-27B-FP8", api_key="nokey", base_url="https://100.108.110.17:8000/v1")

if llm is not None:
    print("LLM initialized successfully")
else:
    print("Failed to initialize LLM")

LLM initialized successfully


In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="Qwen/Qwen3.6-27B-FP8",
    base_url="http://100.108.110.17:8000/v1",
    api_key="nokey",
    temperature=0,
    timeout=60,
    max_retries=1,
)

response = llm.invoke("Hello, how are you?")
print(response.content)

In [10]:
llm_response = llm.invoke("Hello, how are you?")
print(llm_response)

APITimeoutError: Request timed out.

In [5]:
from typing import List
from pydantic import BaseModel, Field


class TaskPlan(BaseModel):
    title: str
    description: str
    priority: str
    estimated_hours: float
    role: str
    dependencies: List[str] = Field(default_factory=list)
    acceptance_criteria: List[str] = Field(default_factory=list)


class MilestonePlan(BaseModel):
    name: str
    goal: str
    estimated_days: int
    deliverables: List[str] = Field(default_factory=list)
    tasks: List[TaskPlan] = Field(default_factory=list)


class ProjectPlan(BaseModel):
    project_name: str
    summary: str
    assumptions: List[str] = Field(default_factory=list)
    milestones: List[MilestonePlan] = Field(default_factory=list)

In [6]:
PLANNING_SYSTEM_PROMPT = """
Bạn là Senior Project Manager.

Hãy tạo project plan theo ĐÚNG schema sau:

ProjectPlan:
- project_name: string
- summary: string
- assumptions: list string
- milestones: list MilestonePlan

MilestonePlan:
- name: string
- goal: string
- estimated_days: integer
- deliverables: list string
- tasks: list TaskPlan

TaskPlan:
- title: string
- description: string
- priority: low | medium | high
- estimated_hours: number
- role: PM | BA | UI/UX Designer | Frontend Developer | Backend Developer | QA Tester | DevOps Engineer | AI Engineer
- dependencies: list string
- acceptance_criteria: list string

Quy tắc:
- Không dùng field "title" cho milestone, phải dùng "name".
- Tasks phải nằm bên trong từng milestone.
- Tối đa 4 milestones.
- Mỗi milestone tối đa 3 tasks.
- Mỗi task description tối đa 1 câu.
- Không markdown.
- Không giải thích ngoài structured output.
"""

In [7]:
parser = JsonOutputParser(pydantic_object=ProjectPlan)

In [8]:
llm = ChatOpenAI(model="claude-haiku-4-5-20251001", 
                temperature=0.2, 
                timeout=60,
                max_retries=3,
                api_key="fe_oa_b963e1daed256e2c821329dd1bac985e2e592148e0fcdbfc", 
                base_url="https://api.freemodel.dev/v1")

In [9]:
structured_output_prompt = llm.with_structured_output(
    ProjectPlan)

In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system", PLANNING_SYSTEM_PROMPT),
    ("human",
    """Yêu cầu tạo dự án : 
    {user_request}
    Hãy tạo project plan chi tiết.""")
])

In [11]:
planning_chain = (
    prompt.partial(format_instructions=parser.get_format_instructions())
    | llm
    | parser
)

In [12]:
user_request = """
Tạo dự án web quản lý phòng thí nghiệm.

Module:
- Quản lý mẫu thí nghiệm
- Đăng ký biểu mẫu
- Quản lý hồ sơ nhân sự
- Hệ thống tài liệu ISO
- Quản lý dịch vụ phân tích mẫu

Yêu cầu:
- Web-based
- Có dashboard
- Có phân quyền
- Có workflow duyệt mẫu
"""

In [ ]:
plan = planning_chain.invoke({"user_request": user_request})

print(json.dumps(
    plan,
    indent=2,
    ensure_ascii=False
))

In [ ]:
for idx, milestone in enumerate(plan["milestones"], start=1):

    print(f"\n=== MILESTONE {idx} ===")

    print("Name:", milestone["name"])

    print("Goal:", milestone["goal"])

    print("Estimated days:", milestone["estimated_days"])

    print("Deliverables:")

    for d in milestone["deliverables"]:
        print("-", d)

NameError: name 'plan' is not defined

In [ ]:
for idx, milestone in enumerate(plan["milestones"], start=1):

    print(f"\n=== MILESTONE {idx} ===")

    print("Name:", milestone["name"])

    print("Goal:", milestone["goal"])

    print("Estimated days:", milestone["estimated_days"])

    print("Deliverables:")

    for d in milestone["deliverables"]:
        print("-", d)


=== MILESTONE 1 ===
Name: Khởi động và phân tích yêu cầu
Goal: Xác định phạm vi, luồng nghiệp vụ, yêu cầu chức năng và phi chức năng cho toàn bộ hệ thống.
Estimated days: 10
Deliverables:
- Tài liệu phạm vi dự án
- Tài liệu yêu cầu nghiệp vụ và chức năng
- Sơ đồ workflow duyệt mẫu
- Danh sách vai trò và ma trận phân quyền

=== MILESTONE 2 ===
Name: Thiết kế hệ thống và giao diện
Goal: Thiết kế kiến trúc giải pháp, cơ sở dữ liệu và giao diện người dùng cho các module chính.
Estimated days: 12
Deliverables:
- Tài liệu kiến trúc hệ thống
- Thiết kế cơ sở dữ liệu
- Wireframe và mockup giao diện
- Prototype dashboard và các màn hình chính

=== MILESTONE 3 ===
Name: Phát triển và kiểm thử hệ thống
Goal: Xây dựng các chức năng cốt lõi, dashboard, phân quyền và workflow duyệt mẫu, đồng thời kiểm thử toàn hệ thống.
Estimated days: 30
Deliverables:
- Ứng dụng web hoàn chỉnh trên môi trường staging
- Các module chức năng theo phạm vi
- Dashboard vận hành
- Báo cáo kiểm thử và danh sách lỗi

=== 

In [1]:
from pathlib import Path

Path("/home/bbsw/agent-pm/init/init.sql")
with open("/home/bbsw/agent-pm/init/init.sql", "r") as f:
    init_sql = f.read()

print(init_sql)


SET statement_timeout = 0;
SET lock_timeout = 0;
SET idle_in_transaction_session_timeout = 0;
SET client_encoding = 'UTF8';
SET standard_conforming_strings = on;
SELECT pg_catalog.set_config('search_path', '', false);
SET check_function_bodies = false;
SET xmloption = content;
SET client_min_messages = warning;
SET row_security = off;

CREATE EXTENSION IF NOT EXISTS pg_trgm WITH SCHEMA public;

-- ---------------------------------------------------------------------------
-- ENUMS
-- ---------------------------------------------------------------------------

-- Vai trò của user trong hệ thống:
--   ADMIN   → toàn quyền cấu hình
--   MANAGER → quản lý project, phê duyệt worklog
--   MEMBER  → log công việc của mình
CREATE TYPE "Role" AS ENUM ('ADMIN', 'MANAGER', 'MEMBER');

-- Vòng đời của một project
CREATE TYPE "ProjectStatus" AS ENUM (
    'PLANNED', 'IN_PROGRESS', 'ON_HOLD', 'COMPLETED', 'CANCELLED'
);

-- Mức độ ưu tiên — dùng chung cho project và task
CREATE TYPE "Priority" AS EN